In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import copy
import math
from tqdm import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

: 

## Table of Contents

- [Problem: AWGN + 2D toy data](#Problem:-AWGN-+-2D-toy-data)

Individual Lifted Methods 
- [Conditional Interpolant](#Conditional-Interpolant). Learns $I_t | F(X) = \alpha_t Z + \beta_t X$
- [Joint Interpolant](#Joint-Interpolant). Learns joint interpolant $I_t = (I_t^{(\mathcal X)} , I_t^{(\mathcal Y)})$, where $I_t^{(\mathcal X)} = \alpha_t Z_{\mathcal X} + \beta X$ and $I_t^{(\mathcal Y)}  = \alpha_t F(X) + \beta Z_{\mathcal Y}$


Comparing methods
- [Comparing Oracle Versions of Conditional & Joint](#Comparing-Oracle-Versions-of-Conditional-&-Joint)
- [Comparing SCSI Versions of Conditional & Joint](#Comparing-SCSI-Versions-of-Conditional-&-Joint)


# Problem: AWGN + 2D toy data

In [ ]:
def forward_corruption(x, noise_std=0.1):
    return x + noise_std * torch.randn_like(x)

def sample_clean(n, kind="two_moons", device=device):
    if kind == "two_moons":
        n1 = n // 2
        n2 = n - n1
        t1 = torch.rand(n1, device=device) * torch.pi
        t2 = torch.rand(n2, device=device) * torch.pi
        a = torch.stack([torch.cos(t1), torch.sin(t1)], dim=1)
        b = torch.stack([1 - torch.cos(t2), 1 - torch.sin(t2) - 0.5], dim=1)
        y = torch.cat([a, b], dim=0) + 0.05 * torch.randn(n, 2, device=device)
        return y[torch.randperm(n, device=device)]
        
    if kind == "checkerboard":  
        y = 4 * torch.rand(n, 2, device=device) - 2
        y[:, 1] += ((torch.floor(y[:, 0]) + torch.floor(y[:, 1])) % 2) * 0.5 - 0.25
        return y
        
    if kind == "gaussian":
        from torch.distributions import MultivariateNormal
        mean = torch.tensor([3.0, 2.0])
        covariance_matrix = torch.tensor([[10, 0.5], 
                                        [0.5, 0.1]])
        dist = MultivariateNormal(mean, covariance_matrix)
        y = dist.sample((n,)).to(device)
        return y

    if kind == "fab_gmm":
        n_mixes = 40
        loc_scaling = 40.0
        log_var_scaling = 1.0
        
        # Use a local generator to keep the 40 mode locations deterministic (seed=0)
        # without affecting the global random state for the actual sampling steps below.
        g = torch.Generator(device=device)
        g.manual_seed(0)
        
        # 1. Initialize component means (locations)
        locs = (torch.rand(n_mixes, 2, generator=g, device=device) - 0.5) * 2 * loc_scaling
        
        # 2. Standard deviation is uniform across all components
        std_dev = torch.exp(torch.tensor(log_var_scaling / 2.0, device=device))
        
        # 3. Uniform mixture probabilities
        probs = torch.ones(n_mixes, device=device) / n_mixes
        
        # 4. Sample the components and the noise
        comp = torch.multinomial(probs, n, replacement=True)
        eps = torch.randn(n, 2, device=device)
        
        # Shift and scale the noise by the chosen component parameters
        y = locs[comp] + std_dev * eps
        return y

    if kind == "dinosaur":
        # Piecewise-linear dinosaur silhouette sampled along segment lengths.
        verts = torch.tensor([
            [-2.20, -0.10], [-1.80,  0.10], [-1.30,  0.35], [-0.70,  0.55],
            [-0.20,  0.70], [ 0.35,  0.80], [ 0.85,  0.75], [ 1.20,  0.55],
            [ 1.45,  0.20], [ 1.55, -0.10], [ 1.40, -0.25], [ 1.10, -0.15],
            [ 1.25,  0.20], [ 1.05,  0.40], [ 0.70,  0.48], [ 0.45,  0.35],
            [ 0.30,  0.10], [ 0.22, -0.22], [ 0.05, -0.62], [-0.10, -0.20],
            [-0.32,  0.08], [-0.60,  0.18], [-0.82,  0.00], [-0.88, -0.35],
            [-0.98, -0.95], [-1.10, -0.35], [-1.35, -0.02], [-1.70, -0.08],
            [-2.20, -0.10],
        ], device=device, dtype=torch.float32)

        p0 = verts[:-1]
        p1 = verts[1:]
        seg = p1 - p0
        seg_len = torch.linalg.norm(seg, dim=1)
        probs = seg_len / seg_len.sum()

        idx = torch.multinomial(probs, n, replacement=True)
        u = torch.rand(n, 1, device=device)
        y = p0[idx] + u * seg[idx]

        # light jitter so points look like a cloud, not exact lines
        jitter = torch.randn(n, 2, device=device) * torch.tensor([0.035, 0.025], device=device)
        y = y + jitter
        return y
    raise ValueError(f"Unknown kind: {kind}")

def sample_corrupted(n, kind="two_moons", noise_std=0.1, device=device):
    y = sample_clean(n, kind, device)
    return forward_corruption(y, noise_std)

try:
    import ot
    def w2_distance(X, Y):
        '''Compute W2 distance using empirical samples'''
        M = ot.dist(X, Y, metric='sqeuclidean')
        a = torch.ones((X.size(0),)) / X.size(0)
        b = torch.ones((Y.size(0),)) / Y.size(0)
        W2_squared = ot.emd2(a, b, M)
        W2 = torch.sqrt(W2_squared)
        return W2
except ImportError:
    print("OT package not found. Wasserstein distance cannot be computed.")
    def w2_distance(X, Y):
        return None

# Conditional Interpolant

Variant of `lifted_scsi_denoiser.ipynb`:

1. **Direct drift parameterization.** The network outputs `b_t(x, y)` directly. No `(1-t)` division anywhere — drift is the primary object.
2. **Time padding.** Training samples `t ~ U(0, 1 - t_eps_train)` and the Heun integrator runs on `t ∈ [0, 1 - t_eps_flow]`. Avoids the irreducible-variance regime near `t=1`.
3. **`y_fake_ratio` knob.** Conditioner is `y_fake = F(X_1(x0))` (Choice 2) for the first `y_fake_ratio` fraction of the batch and real `y` (Choice 1) for the rest, deterministic slice.
4. **`x0_independent` knob.** If `True`, fresh `x0' ⊥ x0` for the interpolant; if `False`, reuse `x0' = x0`. The independent choice avoids the trivial fixed point `x_em = z` ⇒ `b_target = 0` that collapses EM training.

Linear interpolant: `I_t = (1-t)·x0' + t·X_1(x0)`. Drift target:
$$
\mathcal L = \mathbb E\,\|\hat b^\theta_t(I_t, y) - (X_1 - X_0')\|^2.
$$

Trains two models in parallel — `model_oracle` (supervised) and `model_em` (unsupervised EM with the `y_fake_ratio` mix) — sharing `(z, t, x0')` per step.

### SCSI Setup

The network outputs the drift directly: `model(x, y, t) = b_t(x, y)`.

Conditional ODE under the frozen generator `phi_k`:
$$
dX_t = \hat b^\theta_t(X_t, y)\, dt, \qquad X_0 \sim \mathcal N(0, I).
$$

Two models trained in parallel, sharing `(z, t, x0')` per step:

- **`model_oracle`** — supervised. Loss $\mathbb E\,\|\hat b^\theta_t(I_t, y) - (x_{clean} - x_0')\|^2$ on `(x_clean, y_real)` from `sample_clean` + `forward_corruption`.
- **`model_em`** — unsupervised EM with `y_fake_ratio` mix. E-step: `x_em = Φ_{phi_k}(z, y_real)`, `y_fake = F(x_em)`. M-step: regress drift on `(I_t, y_cond, x_em - x0')` where `y_cond` is the deterministic mix.

After each outer iteration `phi_k ← model_em`.

### Visualization

In [ ]:
def visualize_compare(
    model_em,
    model_oracle,
    data_kind,
    noise_std,
    n_vis=1024,
    title_prefix="",
    loss_em=None,
    loss_oracle=None,
):
    em_was_training = model_em.training
    or_was_training = model_oracle.training
    model_em.eval(); model_oracle.eval()
    with torch.no_grad():
        z = torch.randn(n_vis, 2, device=device)
        clean = sample_clean(n_vis, data_kind, device)
        corrupted = forward_corruption(clean, noise_std)
        ode_em = flow(model_em, z, corrupted)
        ode_or = flow(model_oracle, z, corrupted)

    fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))

    all_pts = torch.cat([clean, corrupted, ode_em, ode_or], dim=0).detach().cpu()
    pad = 0.15
    xmin, xmax = all_pts[:, 0].min().item(), all_pts[:, 0].max().item()
    ymin, ymax = all_pts[:, 1].min().item(), all_pts[:, 1].max().item()
    dx, dy = xmax - xmin, ymax - ymin
    xmin, xmax = xmin - pad * dx, xmax + pad * dx
    ymin, ymax = ymin - pad * dy, ymax + pad * dy

    titles = ("Clean", "Corrupted", "EM ODE flow", "Oracle ODE flow")
    datasets = (clean, corrupted, ode_em, ode_or)
    for ax, title, data in zip(axes[:4], titles, datasets):
        x, y = data[:, 0].detach().cpu(), data[:, 1].detach().cpu()
        ax.scatter(x, y, s=8, alpha=0.7)
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("$x_1$")
    axes[0].set_ylabel("$x_2$")

    loss_ax = axes[4]
    loss_ax.set_title("Loss (log-log)")
    if loss_em is not None and len(loss_em) > 0:
        steps = torch.arange(1, len(loss_em) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_em, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="EM", color="tab:blue", alpha=0.8)
    if loss_oracle is not None and len(loss_oracle) > 0:
        steps = torch.arange(1, len(loss_oracle) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_oracle, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Oracle", color="tab:orange", alpha=0.8)
    loss_ax.set_xlabel("step")
    loss_ax.set_ylabel("loss")
    loss_ax.grid(True, alpha=0.3, which="both")
    loss_ax.legend(loc="upper right")

    if title_prefix:
        fig.suptitle(title_prefix)
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    else:
        fig.tight_layout()

    clear_output(wait=True)
    display(fig)

    if em_was_training: model_em.train()
    if or_was_training: model_oracle.train()

    return fig, axes

## Training

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, x_dim=2, y_dim=2, hidden=128*2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(x_dim + y_dim + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, x_dim)
        )

    def forward(self, x_t, y_cond, t):
        return self.net(torch.cat([x_t, y_cond, t], dim=1))

def drift(model_fixed, x, y, t):
    # Direct drift parameterization — the network output IS b_t(x, y).
    return model_fixed(x, y, t)


def flow(model_fixed, z, y, n_steps=64, t_eps=0.0):
    '''Ralston's RK3 integrator on t in [0, 1 - t_eps].

    Stages:  k1 at (t, x)
             k2 at (t + dt/2, x + (dt/2) k1)
             k3 at (t + 3 dt/4, x + (3 dt/4) k2)
    Update:  x_{n+1} = x + dt * (2/9 k1 + 1/3 k2 + 4/9 k3)
    '''
    x = z
    t_final = 1.0 - t_eps
    dt = t_final / n_steps
    for s in range(n_steps):
        t0 = torch.full((z.size(0), 1), s * dt, device=z.device)
        k1 = drift(model_fixed, x, y, t0)
        t_mid = t0 + 0.5 * dt
        k2 = drift(model_fixed, x + 0.5 * dt * k1, y, t_mid)
        t_3q = t0 + 0.75 * dt
        k3 = drift(model_fixed, x + 0.75 * dt * k2, y, t_3q)
        x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
        x = x_new
    if (
        torch.isnan(x_new).any()
        or torch.isinf(x_new).any()
        or torch.isnan(k1).any()
        or torch.isinf(k1).any()
        or torch.isnan(k2).any()
        or torch.isinf(k2).any()
        or torch.isnan(k3).any()
        or torch.isinf(k3).any()
    ):
        print(f"[Numerical Error @ step {s}]: nan/inf detected in flow()!")
        print(f"x: {x}")
        print(f"k1: {k1}")
        print(f"k2: {k2}")
        print(f"k3: {k3}")
        print(f"x_new: {x_new}")
    
    return x

In [ ]:
# Problem Parameters
data_kind = "two_moons"  # "two_moons" or "checkerboard"
noise_std = 0.1

# EM-only knobs
y_fake_ratio = 0.9       # Fraction of EM batch conditioned on y_fake (Choice 2). 1.0 = pure Choice 2.
x0_independent = True    # True: x0' ⊥ x0. False: x0' = x0 (collapses EM to b ≡ 0).
n_steps = 32

# SCSI Parameters
t_outer, t_inner = 5_000, 10

# Training Parameters
batch_size = 516
max_grad_norm = 1.0
base_lr = 1e-4
min_lr = 1e-6

# Logging Parameters
print_every = 100       # Print loss every N global steps (0 to disable).
sample_every = 1000      # Refresh visualization every N global steps.
n_vis = 1024

# Two models, shared hyperparameters.
model_em = SimpleMLP(hidden=516).to(device)
model_oracle = SimpleMLP(hidden=516).to(device)

opt_em = torch.optim.Adam(model_em.parameters(), lr=base_lr)
opt_oracle = torch.optim.Adam(model_oracle.parameters(), lr=base_lr)

total_updates = t_outer * t_inner
sched_em = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_em, T_max=max(total_updates, 1), eta_min=min_lr
)
sched_oracle = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_oracle, T_max=max(total_updates, 1), eta_min=min_lr
)

global_step = 0
loss_history_em = []
loss_history_oracle = []

In [ ]:
phi_k = copy.deepcopy(model_em).eval()

for k in range(t_outer):
    for i in range(t_inner):
        # Shared randomness across the two branches.
        z = torch.randn(batch_size, 2, device=device)
        t = torch.rand(batch_size, 1, device=device)                          # t in [0, 1), no clamp
        z_prime = torch.randn(batch_size, 2, device=device) if x0_independent else z

        # ---- Oracle branch (supervised) ----
        model_oracle.train()
        opt_oracle.zero_grad()
        x_clean = sample_clean(batch_size, data_kind, device)
        y_oracle = forward_corruption(x_clean, noise_std)
        I_t_or = (1 - t) * z_prime + t * x_clean
        b_target_or = x_clean - z_prime
        b_hat_or = model_oracle(I_t_or, y_oracle, t)
        loss_or = ((b_hat_or - b_target_or) ** 2).mean()
        loss_or.backward()
        torch.nn.utils.clip_grad_norm_(model_oracle.parameters(), max_grad_norm)
        opt_oracle.step()
        sched_oracle.step()
        loss_history_oracle.append(loss_or.item())

        # ---- EM branch (unsupervised) ----
        model_em.train()
        opt_em.zero_grad()
        y_real = sample_corrupted(batch_size, data_kind, noise_std, device)
        with torch.no_grad():
            x_em = flow(phi_k, z, y_real, n_steps=n_steps)
            y_fake = forward_corruption(x_em, noise_std)
        n_fake = int(batch_size * y_fake_ratio)
        n_fake = max(0, min(n_fake, batch_size))
        if n_fake == 0:
            y_cond = y_real
        elif n_fake == batch_size:
            y_cond = y_fake
        else:
            y_cond = torch.cat((y_fake[:n_fake], y_real[n_fake:]), dim=0)
        I_t_em = (1 - t) * z_prime + t * x_em
        b_target_em = x_em - z_prime
        b_hat_em = model_em(I_t_em, y_cond, t)
        loss_em_val = ((b_hat_em - b_target_em) ** 2).mean()
        loss_em_val.backward()
        torch.nn.utils.clip_grad_norm_(model_em.parameters(), max_grad_norm)
        opt_em.step()
        sched_em.step()
        loss_history_em.append(loss_em_val.item())

        global_step += 1

        if print_every > 0 and global_step % print_every == 0:
            current_lr = opt_em.param_groups[0]["lr"]
            print(
                f"step={global_step} (outer={k + 1}/{t_outer}) "
                f"loss_em={loss_em_val.item():.6f} loss_or={loss_or.item():.6f} "
                f"lr={current_lr:.2e}"
            )

        if sample_every > 0 and global_step % sample_every == 0:
            visualize_compare(
                model_em,
                model_oracle,
                data_kind,
                noise_std,
                title_prefix=(
                    f"step={global_step} "
                    f"(outer={k + 1}/{t_outer}, inner={i + 1}/{t_inner})"
                ),
                n_vis=n_vis,
                loss_em=loss_history_em,
                loss_oracle=loss_history_oracle,
            )

    phi_k = copy.deepcopy(model_em).eval()

## Notes
Most of the computational bottleneck comes from the E-step where we integrate the ODE. A natural idea is to use a consistency model to expedite this output.

# Joint Interpolant
Consider a forward model $\mathcal F: \mathcal X \to \mathcal Y$. You can solve this by learning the interpolant on the joint space $\Omega = \mathcal X \times \mathcal Y$.

Construct the joint interpolant
    $$
    \begin{align}
        I_t = \begin{pmatrix}
            I_t^{(\mathcal X)}\\
            I_T^{(\mathcal Y)}
        \end{pmatrix} = \begin{pmatrix} 
            \alpha_t Z_{\mathcal X} + \beta_t X \\ 
            \alpha_t \mathcal F(X) + \beta_t Z_{\mathcal Y}
        \end{pmatrix} \in \Omega
    \end{align}
    $$
    Define the coordinates as $\omega = (x,y) \in \Omega$. This optimal drift field $b_t: \Omega \to \Omega$ associated with this interpolant is 
    $$
    \begin{align}
        b_t(\omega) = \mathbb E[\dot I_t | I_t = \omega]
    \end{align}
    $$



## Visualization

In [ ]:
def visualize_compare(
    model_em,
    model_oracle,
    data_kind,
    noise_std,
    n_vis=1024,
    title_prefix="",
    loss_em=None,
    loss_oracle=None,
):
    em_was_training = model_em.training
    or_was_training = model_oracle.training
    model_em.eval(); model_oracle.eval()
    with torch.no_grad():
        z = torch.randn(n_vis, 2, device=device)
        clean = sample_clean(n_vis, data_kind, device)
        corrupted = forward_corruption(clean, noise_std)

        initial_state = torch.cat([z, corrupted], dim=1)
        ode_em = flow(model_em, initial_state)
        ode_or = flow(model_oracle, initial_state)

        ode_em = ode_em[:, :2]
        ode_or = ode_or[:, :2]

    fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))

    all_pts = torch.cat([clean, corrupted, ode_em, ode_or], dim=0).detach().cpu()
    pad = 0.15
    xmin, xmax = all_pts[:, 0].min().item(), all_pts[:, 0].max().item()
    ymin, ymax = all_pts[:, 1].min().item(), all_pts[:, 1].max().item()
    dx, dy = xmax - xmin, ymax - ymin
    xmin, xmax = xmin - pad * dx, xmax + pad * dx
    ymin, ymax = ymin - pad * dy, ymax + pad * dy

    titles = ("Clean", "Corrupted", "EM ODE flow", "Oracle ODE flow")
    datasets = (clean, corrupted, ode_em, ode_or)
    for ax, title, data in zip(axes[:4], titles, datasets):
        x, y = data[:, 0].detach().cpu(), data[:, 1].detach().cpu()
        ax.scatter(x, y, s=8, alpha=0.7)
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("$x_1$")
    axes[0].set_ylabel("$x_2$")

    loss_ax = axes[4]
    loss_ax.set_title("Loss (log-log)")
    if loss_em is not None and len(loss_em) > 0:
        steps = torch.arange(1, len(loss_em) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_em, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="EM", color="tab:blue", alpha=0.8)
    if loss_oracle is not None and len(loss_oracle) > 0:
        steps = torch.arange(1, len(loss_oracle) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_oracle, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Oracle", color="tab:orange", alpha=0.8)
    loss_ax.set_xlabel("step")
    loss_ax.set_ylabel("loss")
    loss_ax.grid(True, alpha=0.3, which="both")
    loss_ax.legend(loc="upper right")

    if title_prefix:
        fig.suptitle(title_prefix)
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    else:
        fig.tight_layout()

    clear_output(wait=True)
    display(fig)

    if em_was_training: model_em.train()
    if or_was_training: model_oracle.train()

    return fig, axes

## Training

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, omega_dim=4, hidden=256, n_freqs=16):
        super().__init__()
        # Fourier features for t
        self.register_buffer("freqs", 2 * math.pi * torch.arange(1, n_freqs + 1).float())
        t_dim = 2 * n_freqs
        self.net = nn.Sequential(
            nn.Linear(omega_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),  # one more layer; cheap insurance
            nn.Linear(hidden, omega_dim),
        )

    def forward(self, omega_t, t):
        tf = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(torch.cat([omega_t, tf], dim=-1))

def drift(model_fixed, x, t):
    # Direct drift parameterization — the network output IS b_t(x, y).
    return model_fixed(x, t)


def flow(model_fixed, z, n_steps=64, t_eps=0.0):
    '''Ralston's RK3 integrator on t in [0, 1 - t_eps].

    Stages:  k1 at (t, x)
             k2 at (t + dt/2, x + (dt/2) k1)
             k3 at (t + 3 dt/4, x + (3 dt/4) k2)
    Update:  x_{n+1} = x + dt * (2/9 k1 + 1/3 k2 + 4/9 k3)
    '''
    x = z
    t_final = 1.0 - t_eps
    dt = t_final / n_steps
    for s in range(n_steps):
        t0 = torch.full((z.size(0), 1), s * dt, device=z.device)
        k1 = drift(model_fixed, x, t0)
        t_mid = t0 + 0.5 * dt
        k2 = drift(model_fixed, x + 0.5 * dt * k1, t_mid)
        t_3q = t0 + 0.75 * dt
        k3 = drift(model_fixed, x + 0.75 * dt * k2, t_3q)
        x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
        x = x_new
    if (
        torch.isnan(x_new).any()
        or torch.isinf(x_new).any()
        or torch.isnan(k1).any()
        or torch.isinf(k1).any()
        or torch.isnan(k2).any()
        or torch.isinf(k2).any()
        or torch.isnan(k3).any()
        or torch.isinf(k3).any()
    ):
        print(f"[Numerical Error @ step {s}]: nan/inf detected in flow()!")
        print(f"x: {x}")
        print(f"k1: {k1}")
        print(f"k2: {k2}")
        print(f"k3: {k3}")
        print(f"x_new: {x_new}")
    return x

In [ ]:
# Problem Parameters
data_kind = "two_moons"  # "two_moons" or "checkerboard"
noise_std = 0.1

# EM-only knobs
y_fake_ratio = 0.9       # Fraction of EM batch conditioned on y_fake (Choice 2). 1.0 = pure Choice 2.
x0_independent = True    # True: x0' ⊥ x0. False: x0' = x0 (collapses EM to b ≡ 0).
n_steps = 32

# SCSI Parameters
t_outer, t_inner = 5_000, 50

# Training Parameters
batch_size = 516
max_grad_norm = 1.0
base_lr = 1e-3
min_lr = 1e-6

# Logging Parameters
print_every = 100       # Print loss every N global steps (0 to disable).
sample_every = 1000      # Refresh visualization every N global steps.
n_vis = 1024

# Two models, shared hyperparameters.
model_em = SimpleMLP(hidden=516).to(device)
model_oracle = SimpleMLP(hidden=516).to(device)

opt_em = torch.optim.Adam(model_em.parameters(), lr=base_lr)
opt_oracle = torch.optim.Adam(model_oracle.parameters(), lr=base_lr)

total_updates = t_outer * t_inner
sched_em = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_em, T_max=max(total_updates, 1), eta_min=min_lr
)
sched_oracle = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_oracle, T_max=max(total_updates, 1), eta_min=min_lr
)

global_step = 0
loss_history_em = []
loss_history_oracle = []

In [ ]:
phi_k = copy.deepcopy(model_em).eval()

for k in range(t_outer):
    for i in range(t_inner):
        # Shared randomness across the two branches.
        z_x = torch.randn(batch_size, 2, device=device)
        z_y = torch.randn(batch_size, 2, device=device)
        t = torch.rand(batch_size, 1, device=device)                          # t in [0, 1), no clamp
        z_prime = torch.randn(batch_size, 2, device=device) if x0_independent else z_x

        # ---- Oracle branch (supervised) ----
        model_oracle.train()
        opt_oracle.zero_grad()
        x_clean = sample_clean(batch_size, data_kind, device)
        y_oracle = forward_corruption(x_clean, noise_std)
        
        I_0_or = torch.cat((z_x, y_oracle), dim=1)
        I_1_or = torch.cat((x_clean, z_y), dim=1)
        I_t_or = (1 - t) * I_0_or + t * I_1_or

        b_target_or = I_1_or - I_0_or
        b_hat_or = model_oracle(I_t_or, t)
        loss_or = ((b_hat_or - b_target_or) ** 2).mean()
        loss_or.backward()
        torch.nn.utils.clip_grad_norm_(model_oracle.parameters(), max_grad_norm)
        opt_oracle.step()
        sched_oracle.step()
        loss_history_oracle.append(loss_or.item())

        # ---- EM branch (unsupervised) ----
        model_em.train()
        opt_em.zero_grad()
        y_real = sample_corrupted(batch_size, data_kind, noise_std, device)
        with torch.no_grad():
            initial_state = torch.cat((z_prime, y_real), dim=1)
            final_state = flow(phi_k, initial_state, n_steps=n_steps) # (batch, 4)
            x_em = final_state[:, :2]
            y_fake = forward_corruption(x_em, noise_std)
        n_fake = int(batch_size * y_fake_ratio)
        n_fake = max(0, min(n_fake, batch_size))
        if n_fake == 0:
            y_cond = y_real
        elif n_fake == batch_size:
            y_cond = y_fake
        else:
            y_cond = torch.cat((y_fake[:n_fake], y_real[n_fake:]), dim=0)
        
        I_0_em = torch.cat((z_x, y_cond), dim=1)
        I_1_em = torch.cat((x_em, z_y), dim=1)
        I_t_em = (1 - t) * I_0_em + t * I_1_em
        
        b_target_em = I_1_em - I_0_em
        b_hat_em = model_em(I_t_em, t)
        loss_em_val = ((b_hat_em - b_target_em) ** 2).mean()
        loss_em_val.backward()
        torch.nn.utils.clip_grad_norm_(model_em.parameters(), max_grad_norm)
        opt_em.step()
        sched_em.step()
        loss_history_em.append(loss_em_val.item())

        global_step += 1

        if print_every > 0 and global_step % print_every == 0:
            current_lr = opt_em.param_groups[0]["lr"]
            print(
                f"step={global_step} (outer={k + 1}/{t_outer}) "
                f"loss_em={loss_em_val.item():.6f} loss_or={loss_or.item():.6f} "
                f"lr={current_lr:.2e}"
            )

        if sample_every > 0 and global_step % sample_every == 0:
            visualize_compare(
                model_em,
                model_oracle,
                data_kind,
                noise_std,
                title_prefix=(
                    f"step={global_step} "
                    f"(outer={k + 1}/{t_outer}, inner={i + 1}/{t_inner})"
                ),
                n_vis=n_vis,
                loss_em=loss_history_em,
                loss_oracle=loss_history_oracle,
            )

    phi_k = copy.deepcopy(model_em).eval()

# Comparing Oracle Versions of Conditional & Joint

In [ ]:
def visualize_compare(
    model_conditional,
    model_joint,
    data_kind,
    noise_std,
    n_vis=1024,
    title_prefix="",
    loss_conditional=None,
    loss_joint=None,
    w2_conditional=None,
    w2_joint=None,
):
    conditional_was_training = model_conditional.training
    joint_was_training = model_joint.training
    model_conditional.eval(); model_joint.eval()
    with torch.no_grad():
        # Sample data
        z = torch.randn(n_vis, 2, device=device)
        clean = sample_clean(n_vis, data_kind, device)
        corrupted = forward_corruption(clean, noise_std)

        # Run ODE
        ode_conditional = flow(model_conditional, z, y_cond=corrupted)

        initial_state = torch.cat([z, corrupted], dim=1)
        ode_joint = flow(model_joint, initial_state)

        # Standardize output of ODE flow, grab only cleaned `x`
        ode_joint = ode_joint[:, :2]

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))

    all_pts = torch.cat([clean, corrupted, ode_conditional, ode_joint], dim=0).detach().cpu()
    pad = 0.15
    xmin, xmax = all_pts[:, 0].min().item(), all_pts[:, 0].max().item()
    ymin, ymax = all_pts[:, 1].min().item(), all_pts[:, 1].max().item()
    dx, dy = xmax - xmin, ymax - ymin
    xmin, xmax = xmin - pad * dx, xmax + pad * dx
    ymin, ymax = ymin - pad * dy, ymax + pad * dy

    scatter_specs = (
        (axes[0, 0], "Clean", clean),
        (axes[1, 0], "Corrupted", corrupted),
        (axes[0, 1], "Conditional ODE flow", ode_conditional),
        (axes[1, 1], "Joint ODE flow", ode_joint),
    )
    for ax, title, data in scatter_specs:
        x, y = data[:, 0].detach().cpu(), data[:, 1].detach().cpu()
        ax.scatter(x, y, s=8, alpha=0.7)
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("$x_1$")
        ax.set_ylabel("$x_2$")

    loss_ax = axes[0, 2]
    loss_ax.set_title("Loss (log-log)")
    if loss_conditional is not None and len(loss_conditional) > 0:
        steps = torch.arange(1, len(loss_conditional) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_conditional, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Conditional", color="tab:blue", alpha=0.8)
    if loss_joint is not None and len(loss_joint) > 0:
        steps = torch.arange(1, len(loss_joint) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_joint, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Joint", color="tab:orange", alpha=0.8)
    loss_ax.set_xlabel("step")
    loss_ax.set_ylabel("loss")
    loss_ax.grid(True, alpha=0.3, which="both")
    loss_ax.legend(loc="upper right")

    w2_ax = axes[1, 2]
    w2_ax.set_title("W2 distance")
    if w2_conditional is not None and len(w2_conditional) > 0:
        steps = torch.arange(1, len(w2_conditional) + 1, device="cpu").numpy()
        values = torch.tensor(w2_conditional, dtype=torch.float32).clamp_min(1e-12).numpy()
        w2_ax.plot(steps, values, linewidth=1.2, label="Conditional", color="tab:blue", alpha=0.8)
    if w2_joint is not None and len(w2_joint) > 0:
        steps = torch.arange(1, len(w2_joint) + 1, device="cpu").numpy()
        values = torch.tensor(w2_joint, dtype=torch.float32).clamp_min(1e-12).numpy()
        w2_ax.plot(steps, values, linewidth=1.2, label="Joint", color="tab:orange", alpha=0.8)
    w2_ax.set_xlabel("step")
    w2_ax.set_ylabel("W2")
    w2_ax.grid(True, alpha=0.3)
    w2_ax.legend(loc="upper right")

    if title_prefix:
        fig.suptitle(title_prefix)
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    else:
        fig.tight_layout()

    clear_output(wait=True)
    display(fig)

    if conditional_was_training:
        model_conditional.train()
    if joint_was_training:
        model_joint.train()

    return fig, axes

In [ ]:
class SimpleMLPJoint(nn.Module):
    def __init__(self, omega_dim=4, hidden=256, n_freqs=16):
        super().__init__()
        # Fourier features for t
        self.register_buffer("freqs", 2 * math.pi * torch.arange(1, n_freqs + 1).float())
        t_dim = 2 * n_freqs
        self.net = nn.Sequential(
            nn.Linear(omega_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),  # one more layer; cheap insurance
            nn.Linear(hidden, omega_dim),
        )

    def forward(self, omega_t, t):
        tf = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(torch.cat([omega_t, tf], dim=-1))

class SimpleMLPConditional(nn.Module):
    def __init__(self, x_dim=2, y_dim=2, hidden=256, n_freqs=16):
        super().__init__()
        # Fourier features for t
        self.register_buffer("freqs", 2 * math.pi * torch.arange(1, n_freqs + 1).float())
        t_dim = 2 * n_freqs
        self.net = nn.Sequential(
            nn.Linear(x_dim + y_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),  # one more layer; cheap insurance
            nn.Linear(hidden, x_dim),
        )

    def forward(self, x, y, t):
        tf = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(torch.cat([x, y, tf], dim=-1))

def drift(model_fixed, x, t, y_cond = None):
    # Direct drift parameterization — the network output IS b_t(x, y).
    if y_cond is not None:
        return model_fixed(x, y_cond, t)
    else:
        return model_fixed(x, t)


def flow(model_fixed, inital_state, n_steps=64, t_eps=0.0, y_cond = None):
    '''Ralston's RK3 integrator on t in [0, 1 - t_eps].

    Stages:  k1 at (t, x)
             k2 at (t + dt/2, x + (dt/2) k1)
             k3 at (t + 3 dt/4, x + (3 dt/4) k2)
    Update:  x_{n+1} = x + dt * (2/9 k1 + 1/3 k2 + 4/9 k3)
    '''
    x = inital_state
    t_final = 1.0 - t_eps
    dt = t_final / n_steps
    if y_cond is not None: #Using conditional model
        for s in range(n_steps):
            t0 = torch.full((inital_state.size(0), 1), s * dt, device=inital_state.device)
            k1 = drift(model_fixed, x, t0, y_cond)
            t_mid = t0 + 0.5 * dt
            k2 = drift(model_fixed, x + 0.5 * dt * k1, t_mid, y_cond)
            t_3q = t0 + 0.75 * dt
            k3 = drift(model_fixed, x + 0.75 * dt * k2, t_3q, y_cond)
            x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
            x = x_new
    else:
        for s in range(n_steps):
            t0 = torch.full((inital_state.size(0), 1), s * dt, device=inital_state.device)
            k1 = drift(model_fixed, x, t0)
            t_mid = t0 + 0.5 * dt
            k2 = drift(model_fixed, x + 0.5 * dt * k1, t_mid)
            t_3q = t0 + 0.75 * dt
            k3 = drift(model_fixed, x + 0.75 * dt * k2, t_3q)
            x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
            x = x_new
    if (
        torch.isnan(x_new).any()
        or torch.isinf(x_new).any()
        or torch.isnan(k1).any()
        or torch.isinf(k1).any()
        or torch.isnan(k2).any()
        or torch.isinf(k2).any()
        or torch.isnan(k3).any()
        or torch.isinf(k3).any()
    ):
        print(f"[Numerical Error @ step {s}]: nan/inf detected in flow()!")
        print(f"x: {x}")
        print(f"k1: {k1}")
        print(f"k2: {k2}")
        print(f"k3: {k3}")
        print(f"x_new: {x_new}")
    return x

In [ ]:
# Problem Parameters
data_kind = "two_moons"  # "two_moons", "checkerboard", "gaussian", "fab_gmm", "dinosaur"
noise_std = .1

# Training Parameters
t_outer = 40_000
batch_size = 516
max_grad_norm = 1.0
base_lr = 1e-3
min_lr = 1e-4

# Logging Parameters
print_every = 100       # Print loss every N global steps (0 to disable).
sample_every = 500      # Refresh visualization every N global steps.
n_vis = 2048

# Two models, shared hyperparameters.
model_conditional = SimpleMLPConditional(hidden=1024).to(device)
model_joint = SimpleMLPJoint(hidden=1024).to(device)

opt_cond = torch.optim.Adam(model_conditional.parameters(), lr=base_lr)
opt_joint = torch.optim.Adam(model_joint.parameters(), lr=base_lr)

sched_cond = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_cond, T_max=max(t_outer, 1), eta_min=min_lr
)
sched_joint = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_joint, T_max=max(t_outer, 1), eta_min=min_lr
)

global_step = 0
loss_history_conditional = []
loss_history_joint = []
loss_history_conditional_warmup = []
loss_history_joint_warmup = []
w2_history_conditional = []
w2_history_joint = []

In [ ]:
t_warmup = 20_000

# ---- Phase 1: Warmup model to prior \pi^(0) = μ (corrupted) ----
for k in tqdm(range(t_warmup)):
    z_x = torch.randn(batch_size, 2, device=device)
    z_y = torch.randn(batch_size, 2, device=device)
    t = torch.rand(batch_size, 1, device=device)                          # t in [0, 1), no clamp
    x_clean = forward_corruption(sample_clean(batch_size, data_kind, device), noise_std)
    
    # ---- Conditional Interpolant, Oracle branch (supervised) ----
    model_conditional.train()
    opt_cond.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    I_t_or = (1 - t) * z_x + t * x_clean
    b_target_or = x_clean - z_x

    b_hat_or = model_conditional(I_t_or, y_oracle, t)
    loss_or = ((b_hat_or - b_target_or) ** 2).mean()
    loss_or.backward()
    torch.nn.utils.clip_grad_norm_(model_conditional.parameters(), max_grad_norm)
    opt_cond.step()
    sched_cond.step()
    loss_history_conditional_warmup.append(loss_or.item())

    # ---- Joint Interpolant, Oracle branch (supervised) ----
    model_joint.train()
    opt_joint.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    
    I_0_or = torch.cat((z_x, y_oracle), dim=1)
    I_1_or = torch.cat((x_clean, z_y), dim=1)
    I_t_or = (1 - t) * I_0_or + t * I_1_or

    b_target_or = I_1_or - I_0_or
    b_hat_or = model_joint(I_t_or, t)
    loss_or = ((b_hat_or - b_target_or) ** 2).mean()
    loss_or.backward()
    torch.nn.utils.clip_grad_norm_(model_joint.parameters(), max_grad_norm)
    opt_joint.step()
    sched_joint.step()
    loss_history_joint_warmup.append(loss_or.item())

    global_step += 1


    if print_every > 0 and global_step % print_every == 0:
        current_lr = opt_joint.param_groups[0]["lr"]
        print(
            f"step={global_step} (outer={k + 1}/{t_outer}) "
            f"loss_cond={loss_history_conditional_warmup[-1]:.6f} loss_joint={loss_history_joint_warmup[-1]:.6f} "
            f"lr={current_lr:.2e}"
        )

    if sample_every > 0 and global_step % 1000 == 0:
        visualize_compare(
            model_conditional,
            model_joint,
            data_kind,
            noise_std,
            title_prefix=(
                f"Comparing Methods in Oracle (supervised) setting. Step={global_step}"
            ),
            n_vis=n_vis,
            loss_conditional=loss_history_conditional_warmup,
            loss_joint=loss_history_joint_warmup,
            w2_conditional=w2_history_conditional,
            w2_joint=None,
        )



In [ ]:
sched_cond = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_cond, T_max=max(t_outer, 1), eta_min=min_lr
)
sched_joint = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_joint, T_max=max(t_outer, 1), eta_min=min_lr
)
# ---- Phase 2: Solving inverse problem ----
for k in tqdm(range(t_outer)):
    # Shared randomness across the two branches.
    z_x = torch.randn(batch_size, 2, device=device)
    z_y = torch.randn(batch_size, 2, device=device)
    t = torch.rand(batch_size, 1, device=device)                          # t in [0, 1), no clamp
    x_clean = sample_clean(batch_size, data_kind, device)


    # ---- Conditional Interpolant, Oracle branch (supervised) ----
    model_conditional.train()
    opt_cond.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    I_t_or = (1 - t) * z_x + t * x_clean
    b_target_or = x_clean - z_x

    b_hat_or = model_conditional(I_t_or, y_oracle, t)
    loss_or = ((b_hat_or - b_target_or) ** 2).mean()
    loss_or.backward()
    torch.nn.utils.clip_grad_norm_(model_conditional.parameters(), max_grad_norm)
    opt_cond.step()
    sched_cond.step()
    loss_history_conditional.append(loss_or.item())
    
    # ---- Joint Interpolant, Oracle branch (supervised) ----
    model_joint.train()
    opt_joint.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    
    I_0_or = torch.cat((z_x, y_oracle), dim=1)
    I_1_or = torch.cat((x_clean, z_y), dim=1)
    I_t_or = (1 - t) * I_0_or + t * I_1_or

    b_target_or = I_1_or - I_0_or
    b_hat_or = model_joint(I_t_or, t)
    loss_or = ((b_hat_or - b_target_or) ** 2).mean()
    loss_or.backward()
    torch.nn.utils.clip_grad_norm_(model_joint.parameters(), max_grad_norm)
    opt_joint.step()
    sched_joint.step()
    loss_history_joint.append(loss_or.item())

    global_step += 1

    if print_every > 0 and global_step % print_every == 0:
        current_lr = opt_joint.param_groups[0]["lr"]
        print(
            f"step={global_step} (outer={k + 1}/{t_outer}) "
            f"loss_cond={loss_history_conditional[-1]:.6f} loss_joint={loss_history_joint[-1]:.6f} "
            f"lr={current_lr:.2e}"
        )

    if sample_every > 0 and global_step % sample_every == 0:
        with torch.no_grad():
            clean_eval = sample_clean(n_vis, data_kind, device)
            corrupted_eval = forward_corruption(clean_eval, noise_std)
            z_eval = torch.randn(n_vis, 2, device=device)

            conditional_eval = flow(model_conditional, z_eval, y_cond=corrupted_eval)
            joint_state_eval = torch.cat([z_eval, corrupted_eval], dim=1)
            joint_eval = flow(model_joint, joint_state_eval)[:, :2]

            w2_cond = w2_distance(conditional_eval, clean_eval)
            w2_joint = w2_distance(joint_eval, clean_eval)

        if w2_cond is not None:
            w2_history_conditional.append(float(w2_cond))
        if w2_joint is not None:
            w2_history_joint.append(float(w2_joint))

        visualize_compare(
            model_conditional,
            model_joint,
            data_kind,
            noise_std,
            title_prefix=(
                f"Comparing Methods in Oracle (supervised) setting. Step={global_step}"
            ),
            n_vis=n_vis,
            loss_conditional=loss_history_conditional,
            loss_joint=loss_history_joint,
            w2_conditional=w2_history_conditional,
            w2_joint=w2_history_joint,
        )



# Comparing SCSI Versions of Conditional & Joint

In [ ]:
def visualize_compare(
    model_conditional,
    model_joint,
    data_kind,
    noise_std,
    n_vis=1024,
    title_prefix="",
    loss_conditional=None,
    loss_joint=None,
    w2_conditional=None,
    w2_joint=None,
):
    conditional_was_training = model_conditional.training
    joint_was_training = model_joint.training
    model_conditional.eval(); model_joint.eval()
    with torch.no_grad():
        # Sample data
        z = torch.randn(n_vis, 2, device=device)
        clean = sample_clean(n_vis, data_kind, device)
        corrupted = forward_corruption(clean, noise_std)

        # Run ODE
        ode_conditional = flow(model_conditional, z, y_cond=corrupted)

        initial_state = torch.cat([z, corrupted], dim=1)
        ode_joint = flow(model_joint, initial_state)

        # Standardize output of ODE flow, grab only cleaned `x`
        ode_joint = ode_joint[:, :2]

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))

    all_pts = torch.cat([clean, corrupted, ode_conditional, ode_joint], dim=0).detach().cpu()
    pad = 0.15
    xmin, xmax = all_pts[:, 0].min().item(), all_pts[:, 0].max().item()
    ymin, ymax = all_pts[:, 1].min().item(), all_pts[:, 1].max().item()
    dx, dy = xmax - xmin, ymax - ymin
    xmin, xmax = xmin - pad * dx, xmax + pad * dx
    ymin, ymax = ymin - pad * dy, ymax + pad * dy

    scatter_specs = (
        (axes[0, 0], "Clean", clean),
        (axes[1, 0], "Corrupted", corrupted),
        (axes[0, 1], "Conditional ODE flow", ode_conditional),
        (axes[1, 1], "Joint ODE flow", ode_joint),
    )
    for ax, title, data in scatter_specs:
        x, y = data[:, 0].detach().cpu(), data[:, 1].detach().cpu()
        ax.scatter(x, y, s=8, alpha=0.7)
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("$x_1$")
        ax.set_ylabel("$x_2$")

    loss_ax = axes[0, 2]
    loss_ax.set_title("Loss (log-log)")
    if loss_conditional is not None and len(loss_conditional) > 0:
        steps = torch.arange(1, len(loss_conditional) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_conditional, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Conditional", color="tab:blue", alpha=0.8)
    if loss_joint is not None and len(loss_joint) > 0:
        steps = torch.arange(1, len(loss_joint) + 1, device="cpu").numpy()
        losses = torch.tensor(loss_joint, dtype=torch.float32).clamp_min(1e-12).numpy()
        loss_ax.loglog(steps, losses, linewidth=1.2, label="Joint", color="tab:orange", alpha=0.8)
    loss_ax.set_xlabel("step")
    loss_ax.set_ylabel("loss")
    loss_ax.grid(True, alpha=0.3, which="both")
    loss_ax.legend(loc="upper right")

    w2_ax = axes[1, 2]
    w2_ax.set_title("W2 distance")
    if w2_conditional is not None and len(w2_conditional) > 0:
        steps = torch.arange(1, len(w2_conditional) + 1, device="cpu").numpy()
        values = torch.tensor(w2_conditional, dtype=torch.float32).clamp_min(1e-12).numpy()
        w2_ax.plot(steps, values, linewidth=1.2, label="Conditional", color="tab:blue", alpha=0.8)
    if w2_joint is not None and len(w2_joint) > 0:
        steps = torch.arange(1, len(w2_joint) + 1, device="cpu").numpy()
        values = torch.tensor(w2_joint, dtype=torch.float32).clamp_min(1e-12).numpy()
        w2_ax.plot(steps, values, linewidth=1.2, label="Joint", color="tab:orange", alpha=0.8)
    w2_ax.set_xlabel("step")
    w2_ax.set_ylabel("W2")
    w2_ax.grid(True, alpha=0.3)
    w2_ax.legend(loc="upper right")

    if title_prefix:
        fig.suptitle(title_prefix)
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    else:
        fig.tight_layout()

    clear_output(wait=True)
    display(fig)

    if conditional_was_training:
        model_conditional.train()
    if joint_was_training:
        model_joint.train()

    return fig, axes

In [ ]:
class SimpleMLPJoint(nn.Module):
    def __init__(self, omega_dim=4, hidden=256, n_freqs=16):
        super().__init__()
        # Fourier features for t
        self.register_buffer("freqs", 2 * math.pi * torch.arange(1, n_freqs + 1).float())
        t_dim = 2 * n_freqs
        self.net = nn.Sequential(
            nn.Linear(omega_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),  # one more layer; cheap insurance
            nn.Linear(hidden, omega_dim),
        )

    def forward(self, omega_t, t):
        tf = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(torch.cat([omega_t, tf], dim=-1))

class SimpleMLPConditional(nn.Module):
    def __init__(self, x_dim=2, y_dim=2, hidden=256, n_freqs=16):
        super().__init__()
        # Fourier features for t
        self.register_buffer("freqs", 2 * math.pi * torch.arange(1, n_freqs + 1).float())
        t_dim = 2 * n_freqs
        self.net = nn.Sequential(
            nn.Linear(x_dim + y_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),  # one more layer; cheap insurance
            nn.Linear(hidden, x_dim),
        )

    def forward(self, x, y, t):
        tf = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        return self.net(torch.cat([x, y, tf], dim=-1))

def drift(model_fixed, x, t, y_cond = None):
    # Direct drift parameterization — the network output IS b_t(x, y).
    if y_cond is not None:
        return model_fixed(x, y_cond, t)
    else:
        return model_fixed(x, t)


def flow(model_fixed, inital_state, n_steps=64, t_eps=0.0, y_cond = None):
    '''Ralston's RK3 integrator on t in [0, 1 - t_eps].

    Stages:  k1 at (t, x)
             k2 at (t + dt/2, x + (dt/2) k1)
             k3 at (t + 3 dt/4, x + (3 dt/4) k2)
    Update:  x_{n+1} = x + dt * (2/9 k1 + 1/3 k2 + 4/9 k3)
    '''
    x = inital_state
    t_final = 1.0 - t_eps
    dt = t_final / n_steps
    if y_cond is not None: #Using conditional model
        for s in range(n_steps):
            t0 = torch.full((inital_state.size(0), 1), s * dt, device=inital_state.device)
            k1 = drift(model_fixed, x, t0, y_cond)
            t_mid = t0 + 0.5 * dt
            k2 = drift(model_fixed, x + 0.5 * dt * k1, t_mid, y_cond)
            t_3q = t0 + 0.75 * dt
            k3 = drift(model_fixed, x + 0.75 * dt * k2, t_3q, y_cond)
            x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
            x = x_new
    else:
        for s in range(n_steps):
            t0 = torch.full((inital_state.size(0), 1), s * dt, device=inital_state.device)
            k1 = drift(model_fixed, x, t0)
            t_mid = t0 + 0.5 * dt
            k2 = drift(model_fixed, x + 0.5 * dt * k1, t_mid)
            t_3q = t0 + 0.75 * dt
            k3 = drift(model_fixed, x + 0.75 * dt * k2, t_3q)
            x_new = x + dt * (2.0 / 9.0 * k1 + 1.0 / 3.0 * k2 + 4.0 / 9.0 * k3)
            x = x_new
    if (
        torch.isnan(x_new).any()
        or torch.isinf(x_new).any()
        or torch.isnan(k1).any()
        or torch.isinf(k1).any()
        or torch.isnan(k2).any()
        or torch.isinf(k2).any()
        or torch.isnan(k3).any()
        or torch.isinf(k3).any()
    ):
        print(f"[Numerical Error @ step {s}]: nan/inf detected in flow()!")
        print(f"x: {x}")
        print(f"k1: {k1}")
        print(f"k2: {k2}")
        print(f"k3: {k3}")
        print(f"x_new: {x_new}")
    return x

In [ ]:
# Problem Parameters
data_kind = "two_moons"
noise_std = .1

# EM-only knobs
y_fake_ratio = 0.9
x0_independent = True
n_steps = 32

# SCSI Parameters
t_outer = 5_000
t_inner = 10

# Training Parameters
batch_size = 516
max_grad_norm = 1.0
base_lr = 1e-3
min_lr = 1e-4

# Logging Parameters
print_every = 100
sample_every = 500
n_vis = 2048

# Two models, shared hyperparameters.
model_conditional = SimpleMLPConditional(hidden=1024).to(device)
model_joint = SimpleMLPJoint(hidden=1024).to(device)

opt_cond = torch.optim.Adam(model_conditional.parameters(), lr=base_lr)
opt_joint = torch.optim.Adam(model_joint.parameters(), lr=base_lr)

total_updates = t_outer * t_inner
sched_cond = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_cond, T_max=max(total_updates, 1), eta_min=min_lr
)
sched_joint = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_joint, T_max=max(total_updates, 1), eta_min=min_lr
)

global_step = 0
loss_history_conditional = []
loss_history_joint = []
w2_history_conditional = []
w2_history_joint = []

In [ ]:
t_warmup = 5_000

# ---- Phase 1: Warmup model to prior π^(0) = μ (corrupted) ----
for k in tqdm(range(t_warmup)):
    z_x = torch.randn(batch_size, 2, device=device)
    z_y = torch.randn(batch_size, 2, device=device)
    t = torch.rand(batch_size, 1, device=device)
    x_clean = forward_corruption(sample_clean(batch_size, data_kind, device), noise_std)

    # ---- Conditional warmup (supervised on corrupted prior) ----
    model_conditional.train()
    opt_cond.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    I_t = (1 - t) * z_x + t * x_clean
    b_target = x_clean - z_x
    b_hat = model_conditional(I_t, y_oracle, t)
    loss_cond = ((b_hat - b_target) ** 2).mean()
    loss_cond.backward()
    torch.nn.utils.clip_grad_norm_(model_conditional.parameters(), max_grad_norm)
    opt_cond.step()
    sched_cond.step()

    # ---- Joint warmup (supervised on corrupted prior) ----
    model_joint.train()
    opt_joint.zero_grad()
    y_oracle = forward_corruption(x_clean, noise_std)
    I_0 = torch.cat((z_x, y_oracle), dim=1)
    I_1 = torch.cat((x_clean, z_y), dim=1)
    I_t_j = (1 - t) * I_0 + t * I_1
    b_target_j = I_1 - I_0
    b_hat_j = model_joint(I_t_j, t)
    loss_joint_w = ((b_hat_j - b_target_j) ** 2).mean()
    loss_joint_w.backward()
    torch.nn.utils.clip_grad_norm_(model_joint.parameters(), max_grad_norm)
    opt_joint.step()
    sched_joint.step()

    global_step += 1

    if print_every > 0 and global_step % print_every == 0:
        current_lr = opt_cond.param_groups[0]["lr"]
        print(
            f"[Warmup] step={global_step}/{t_warmup} "
            f"loss_cond={loss_cond.item():.6f} loss_joint={loss_joint_w.item():.6f} "
            f"lr={current_lr:.2e}"
        )

In [ ]:
sched_cond = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_cond, T_max=max(t_outer * t_inner, 1), eta_min=min_lr
)
sched_joint = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_joint, T_max=max(t_outer * t_inner, 1), eta_min=min_lr
)

phi_k_cond = copy.deepcopy(model_conditional).eval()
phi_k_joint = copy.deepcopy(model_joint).eval()

# ---- Phase 2: SCSI EM ----
for k in tqdm(range(t_outer)):
    for i in range(t_inner):
        z_x = torch.randn(batch_size, 2, device=device)
        z_y = torch.randn(batch_size, 2, device=device)
        t = torch.rand(batch_size, 1, device=device)
        z_prime = torch.randn(batch_size, 2, device=device) if x0_independent else z_x

        # ===== Conditional SCSI =====
        model_conditional.train()
        opt_cond.zero_grad()
        y_real_cond = sample_corrupted(batch_size, data_kind, noise_std, device)
        with torch.no_grad():
            x_em_cond = flow(phi_k_cond, z_x, n_steps=n_steps, y_cond=y_real_cond)
            y_fake_cond = forward_corruption(x_em_cond, noise_std)
        n_fake = int(batch_size * y_fake_ratio)
        n_fake = max(0, min(n_fake, batch_size))
        if n_fake == 0:
            y_cond_mix = y_real_cond
        elif n_fake == batch_size:
            y_cond_mix = y_fake_cond
        else:
            y_cond_mix = torch.cat((y_fake_cond[:n_fake], y_real_cond[n_fake:]), dim=0)
        I_t_cond = (1 - t) * z_prime + t * x_em_cond
        b_target_cond = x_em_cond - z_prime
        b_hat_cond = model_conditional(I_t_cond, y_cond_mix, t)
        loss_cond_val = ((b_hat_cond - b_target_cond) ** 2).mean()
        loss_cond_val.backward()
        torch.nn.utils.clip_grad_norm_(model_conditional.parameters(), max_grad_norm)
        opt_cond.step()
        sched_cond.step()
        loss_history_conditional.append(loss_cond_val.item())

        # ===== Joint SCSI =====
        model_joint.train()
        opt_joint.zero_grad()
        y_real_joint = sample_corrupted(batch_size, data_kind, noise_std, device)
        with torch.no_grad():
            initial_state = torch.cat((z_prime, y_real_joint), dim=1)
            final_state = flow(phi_k_joint, initial_state, n_steps=n_steps)
            x_em_joint = final_state[:, :2]
            y_fake_joint = forward_corruption(x_em_joint, noise_std)
        n_fake_j = int(batch_size * y_fake_ratio)
        n_fake_j = max(0, min(n_fake_j, batch_size))
        if n_fake_j == 0:
            y_cond_joint = y_real_joint
        elif n_fake_j == batch_size:
            y_cond_joint = y_fake_joint
        else:
            y_cond_joint = torch.cat((y_fake_joint[:n_fake_j], y_real_joint[n_fake_j:]), dim=0)
        I_0_em = torch.cat((z_x, y_cond_joint), dim=1)
        I_1_em = torch.cat((x_em_joint, z_y), dim=1)
        I_t_joint = (1 - t) * I_0_em + t * I_1_em
        b_target_joint = I_1_em - I_0_em
        b_hat_joint = model_joint(I_t_joint, t)
        loss_joint_val = ((b_hat_joint - b_target_joint) ** 2).mean()
        loss_joint_val.backward()
        torch.nn.utils.clip_grad_norm_(model_joint.parameters(), max_grad_norm)
        opt_joint.step()
        sched_joint.step()
        loss_history_joint.append(loss_joint_val.item())

        global_step += 1

        if print_every > 0 and global_step % print_every == 0:
            current_lr = opt_cond.param_groups[0]["lr"]
            print(
                f"step={global_step} (outer={k + 1}/{t_outer}, inner={i + 1}/{t_inner}) "
                f"loss_cond={loss_cond_val.item():.6f} loss_joint={loss_joint_val.item():.6f} "
                f"lr={current_lr:.2e}"
            )

        if sample_every > 0 and global_step % sample_every == 0:
            with torch.no_grad():
                clean_eval = sample_clean(n_vis, data_kind, device)
                corrupted_eval = forward_corruption(clean_eval, noise_std)
                z_eval = torch.randn(n_vis, 2, device=device)

                conditional_eval = flow(model_conditional, z_eval, y_cond=corrupted_eval)
                joint_state_eval = torch.cat([z_eval, corrupted_eval], dim=1)
                joint_eval = flow(model_joint, joint_state_eval)[:, :2]

                w2_cond = w2_distance(conditional_eval, clean_eval)
                w2_jnt = w2_distance(joint_eval, clean_eval)

            if w2_cond is not None:
                w2_history_conditional.append(float(w2_cond))
            if w2_jnt is not None:
                w2_history_joint.append(float(w2_jnt))

            visualize_compare(
                model_conditional,
                model_joint,
                data_kind,
                noise_std,
                title_prefix=f"Comparing SCSI Methods. Step={global_step}",
                n_vis=n_vis,
                loss_conditional=loss_history_conditional,
                loss_joint=loss_history_joint,
                w2_conditional=w2_history_conditional,
                w2_joint=w2_history_joint,
            )

    phi_k_cond = copy.deepcopy(model_conditional).eval()
    phi_k_joint = copy.deepcopy(model_joint).eval()